# Crypto Next-Candle Prediction (Colab Research Notebook)

This notebook is a **research + implementation template** for predicting the next candle in trading data with ML/DL.

It covers:
- Multiple data-source options (crypto + stocks/forex)
- Data cleaning, feature engineering, labeling, leakage checks
- Baseline ML models + optional deep learning windowed model
- Time-series evaluation + simple strategy backtest metrics
- Reproducibility best practices for Google Colab

> Educational use only (not financial advice).

## 0) Colab setup
- Runtime: Python 3 (GPU optional for deep learning)
- Suggested: `Runtime -> Change runtime type -> GPU` for LSTM section

In [ ]:
!pip -q install ccxt yfinance ta xgboost scikit-learn pandas numpy matplotlib seaborn joblib

In [ ]:
import platform, sys, warnings, random
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import ccxt
import yfinance as yf

from ta.trend import EMAIndicator, MACD, ADXIndicator, SMAIndicator
from ta.momentum import RSIIndicator, StochasticOscillator
from ta.volatility import BollingerBands, AverageTrueRange
from ta.volume import OnBalanceVolumeIndicator

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    mean_absolute_error, mean_squared_error, r2_score, confusion_matrix,
)
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

import joblib

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

print("Python:", sys.version.split()[0], "| Platform:", platform.platform())

## 1) Research notes: data sources and formats
Common sources you can use in the same notebook:
- **Crypto**: Binance/Coinbase/Bybit via `ccxt`.
- **Stocks/ETFs**: Yahoo Finance via `yfinance`, Alpha Vantage, IEX, Quandl.
- **Forex/Commodities**: OANDA, Finnhub, broker APIs.

Typical formats:
- REST JSON or CSV responses parsed into Pandas DataFrames.
- Direct CSV/Kaggle files.
- Database tables.

This notebook includes two loaders:
1. `ccxt` for crypto OHLCV.
2. `yfinance` for equities/crypto symbols supported by Yahoo.

In [ ]:
# =========================
# User configuration
# =========================
DATA_SOURCE = "ccxt"      # "ccxt" or "yfinance"
SYMBOL = "BTC/USDT"       # ccxt format, e.g. BTC/USDT
YF_TICKER = "BTC-USD"     # yfinance format, e.g. BTC-USD or MSFT
TIMEFRAME = "1h"          # ccxt timeframe
LIMIT = 3000              # ccxt candles
START = "2022-01-01"      # yfinance start
END = None                # yfinance end

# Label settings
DIRECTION_THRESHOLD = 0.0

# Split and modeling
TEST_SIZE = 0.2
N_SPLITS_CV = 5

In [ ]:
def fetch_from_ccxt(symbol="BTC/USDT", timeframe="1h", limit=2000, exchange_id="binance"):
    ex_cls = getattr(ccxt, exchange_id)
    ex = ex_cls({"enableRateLimit": True})
    rows = ex.fetch_ohlcv(symbol, timeframe=timeframe, limit=limit)
    df = pd.DataFrame(rows, columns=["timestamp", "open", "high", "low", "close", "volume"])
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms", utc=True)
    return df.sort_values("timestamp").reset_index(drop=True)


def fetch_from_yfinance(ticker="BTC-USD", start="2022-01-01", end=None, interval="1h"):
    data = yf.download(ticker, start=start, end=end, interval=interval, auto_adjust=False, progress=False)
    data = data.reset_index()
    data.columns = [c.lower().replace(" ", "_") for c in data.columns]
    rename_map = {
        "datetime": "timestamp",
        "date": "timestamp",
        "adj_close": "adj_close",
    }
    for old, new in rename_map.items():
        if old in data.columns:
            data = data.rename(columns={old: new})
    base_cols = ["timestamp", "open", "high", "low", "close", "volume"]
    missing = [c for c in base_cols if c not in data.columns]
    if missing:
        raise ValueError(f"Missing columns from yfinance data: {missing}")
    data["timestamp"] = pd.to_datetime(data["timestamp"], utc=True)
    return data[base_cols].sort_values("timestamp").reset_index(drop=True)


if DATA_SOURCE == "ccxt":
    raw_df = fetch_from_ccxt(SYMBOL, TIMEFRAME, LIMIT)
elif DATA_SOURCE == "yfinance":
    # yfinance intervals like 1m/5m/15m/30m/60m/1d etc.
    yf_interval = "60m" if TIMEFRAME == "1h" else TIMEFRAME
    raw_df = fetch_from_yfinance(YF_TICKER, START, END, yf_interval)
else:
    raise ValueError("DATA_SOURCE must be 'ccxt' or 'yfinance'")

print(raw_df.shape)
raw_df.head()

## 2) Preprocessing
- Sort timestamps, drop duplicates.
- Fill missing values (ffill/bfill).
- Basic sanity checks.

In [ ]:
def preprocess_ohlcv(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy().sort_values("timestamp").drop_duplicates("timestamp")

    # Replace impossible rows and fill missing values
    for c in ["open", "high", "low", "close", "volume"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.ffill().bfill()

    # Optional: enforce basic candle consistency
    df["high"] = df[["open", "high", "low", "close"]].max(axis=1)
    df["low"] = df[["open", "high", "low", "close"]].min(axis=1)
    return df.reset_index(drop=True)


df = preprocess_ohlcv(raw_df)
print(df.isna().sum().sum(), "total missing values after preprocessing")
df.head(3)

## 3) Feature engineering
Includes:
- returns and candle geometry (body/wicks/range)
- indicators (SMA/EMA, RSI, MACD, Bollinger, ATR, ADX, Stochastic)
- lagged features

In [ ]:
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()

    # Returns
    d["ret_1"] = d["close"].pct_change(1)
    d["ret_3"] = d["close"].pct_change(3)
    d["ret_6"] = d["close"].pct_change(6)

    # Candle geometry
    d["body"] = d["close"] - d["open"]
    d["body_abs"] = d["body"].abs()
    d["upper_wick"] = d["high"] - d[["open", "close"]].max(axis=1)
    d["lower_wick"] = d[["open", "close"]].min(axis=1) - d["low"]
    d["range"] = d["high"] - d["low"]
    d["range_pct"] = d["range"] / d["close"].replace(0, np.nan)

    # Simple pattern flags (illustrative)
    d["is_doji"] = (d["body_abs"] <= 0.1 * d["range"].replace(0, np.nan)).astype(int)
    d["is_hammer_like"] = ((d["lower_wick"] > 2 * d["body_abs"]) & (d["upper_wick"] < d["body_abs"])).astype(int)

    # Indicators
    d["sma_20"] = SMAIndicator(d["close"], 20).sma_indicator()
    d["ema_9"] = EMAIndicator(d["close"], 9).ema_indicator()
    d["ema_21"] = EMAIndicator(d["close"], 21).ema_indicator()

    m = MACD(d["close"], 26, 12, 9)
    d["macd"] = m.macd()
    d["macd_signal"] = m.macd_signal()
    d["macd_diff"] = m.macd_diff()

    d["rsi_14"] = RSIIndicator(d["close"], 14).rsi()

    st = StochasticOscillator(d["high"], d["low"], d["close"], window=14, smooth_window=3)
    d["stoch_k"] = st.stoch()
    d["stoch_d"] = st.stoch_signal()

    bb = BollingerBands(d["close"], window=20, window_dev=2)
    d["bb_high"] = bb.bollinger_hband()
    d["bb_low"] = bb.bollinger_lband()
    d["bb_mid"] = bb.bollinger_mavg()
    d["bb_width"] = bb.bollinger_wband()

    d["atr_14"] = AverageTrueRange(d["high"], d["low"], d["close"], window=14).average_true_range()
    d["adx_14"] = ADXIndicator(d["high"], d["low"], d["close"], window=14).adx()
    d["obv"] = OnBalanceVolumeIndicator(d["close"], d["volume"]).on_balance_volume()

    # Lags
    for lag in [1,2,3,6,12,24]:
        d[f"close_lag_{lag}"] = d["close"].shift(lag)
        d[f"vol_lag_{lag}"] = d["volume"].shift(lag)

    return d


feat_df = add_features(df)
feat_df.head(3)

## 4) Labeling (targets)
- Regression target: next close
- Classification target: next return above threshold

No look-ahead leakage in features: labels are shifted **forward** (`shift(-1)`).

In [ ]:
def add_targets(d: pd.DataFrame, threshold=0.0) -> pd.DataFrame:
    out = d.copy()
    out["target_close_next"] = out["close"].shift(-1)
    out["target_return_next"] = out["close"].shift(-1) / out["close"] - 1
    out["target_up_next"] = (out["target_return_next"] > threshold).astype(int)
    return out

model_df = add_targets(feat_df, DIRECTION_THRESHOLD)
model_df = model_df.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)
print(model_df.shape)
model_df[["timestamp","close","target_close_next","target_return_next","target_up_next"]].head()

## 5) Train/test split (chronological)
Never random-shuffle time series.

In [ ]:
exclude_cols = ["timestamp", "target_close_next", "target_return_next", "target_up_next"]
feature_cols = [c for c in model_df.columns if c not in exclude_cols]

X = model_df[feature_cols]
y_reg = model_df["target_close_next"]
y_cls = model_df["target_up_next"]

split_idx = int(len(model_df) * (1 - TEST_SIZE))
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_reg_train, y_reg_test = y_reg.iloc[:split_idx], y_reg.iloc[split_idx:]
y_cls_train, y_cls_test = y_cls.iloc[:split_idx], y_cls.iloc[split_idx:]

ts_train = model_df.iloc[:split_idx]["timestamp"]
ts_test = model_df.iloc[split_idx:]["timestamp"]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train/Test:", X_train.shape, X_test.shape)

## 6) Baseline models
- Classification: RandomForest (`target_up_next`)
- Regression: XGBoost + Ridge baseline (`target_close_next`)

In [ ]:
rf_clf = RandomForestClassifier(
    n_estimators=400,
    max_depth=10,
    min_samples_leaf=3,
    random_state=SEED,
    n_jobs=-1,
)
rf_clf.fit(X_train, y_cls_train)
cls_pred = rf_clf.predict(X_test)
cls_prob = rf_clf.predict_proba(X_test)[:, 1]

xgb_reg = XGBRegressor(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="reg:squarederror",
    random_state=SEED,
)
xgb_reg.fit(X_train, y_reg_train)
reg_pred_xgb = xgb_reg.predict(X_test)

ridge_reg = Ridge(alpha=1.0)
ridge_reg.fit(X_train_scaled, y_reg_train)
reg_pred_ridge = ridge_reg.predict(X_test_scaled)

## 7) Metrics
### Classification
- Accuracy, Precision, Recall, F1, ROC AUC

### Regression
- MAE, RMSE, R²

In [ ]:
def regression_metrics(y_true, y_pred, name="model"):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"[{name}] MAE={mae:.4f} | RMSE={rmse:.4f} | R2={r2:.4f}")

print("Classification:")
print("Accuracy:", round(accuracy_score(y_cls_test, cls_pred), 4))
print("Precision:", round(precision_score(y_cls_test, cls_pred, zero_division=0), 4))
print("Recall:", round(recall_score(y_cls_test, cls_pred, zero_division=0), 4))
print("F1:", round(f1_score(y_cls_test, cls_pred, zero_division=0), 4))
print("ROC AUC:", round(roc_auc_score(y_cls_test, cls_prob), 4))
print("Confusion matrix\n", confusion_matrix(y_cls_test, cls_pred))

print("\nRegression:")
regression_metrics(y_reg_test, reg_pred_xgb, "XGBRegressor")
regression_metrics(y_reg_test, reg_pred_ridge, "Ridge")

## 8) Time-series cross-validation (optional but recommended)
Uses `TimeSeriesSplit` to mimic rolling origin evaluation.

In [ ]:
ts_cv = TimeSeriesSplit(n_splits=N_SPLITS_CV)
cv_scores = []

for fold, (tr_idx, va_idx) in enumerate(ts_cv.split(X), start=1):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y_cls.iloc[tr_idx], y_cls.iloc[va_idx]

    model = RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)
    model.fit(X_tr, y_tr)
    pred = model.predict(X_va)
    cv_scores.append(f1_score(y_va, pred, zero_division=0))
    print(f"Fold {fold}: F1={cv_scores[-1]:.4f}")

print("CV F1 mean:", round(float(np.mean(cv_scores)), 4))

## 9) Visualization

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(ts_test.values, y_reg_test.values, label="Actual next close")
plt.plot(ts_test.values, reg_pred_xgb, label="Predicted next close (XGB)", alpha=0.8)
plt.title("Next-candle close prediction")
plt.xlabel("Time")
plt.ylabel("Price")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
feat_imp = pd.Series(rf_clf.feature_importances_, index=feature_cols).sort_values(ascending=False).head(20)
plt.figure(figsize=(8, 6))
feat_imp.sort_values().plot(kind="barh")
plt.title("Top 20 feature importances (RandomForest)")
plt.tight_layout()
plt.show()

## 10) Simple strategy backtest metrics from classification signal
Signal rule (example):
- Long if model predicts UP (1), else flat.
- Strategy return = signal * next_return.

Metrics shown:
- cumulative return
- annualized Sharpe (approx)
- max drawdown

In [ ]:
bt = model_df.iloc[split_idx:].copy().reset_index(drop=True)
bt["pred_up"] = cls_pred
bt["market_ret"] = bt["target_return_next"]
bt["strategy_ret"] = bt["pred_up"] * bt["market_ret"]

bt["equity_market"] = (1 + bt["market_ret"]).cumprod()
bt["equity_strategy"] = (1 + bt["strategy_ret"]).cumprod()

def max_drawdown(equity_curve):
    roll_max = equity_curve.cummax()
    dd = equity_curve / roll_max - 1
    return float(dd.min())

# Approx annualization factor map
ann_factor = {
    "1m": 365*24*60,
    "5m": 365*24*12,
    "15m": 365*24*4,
    "30m": 365*24*2,
    "1h": 365*24,
    "4h": 365*6,
    "1d": 365,
}.get(TIMEFRAME, 365)

strat_mean = bt["strategy_ret"].mean()
strat_std = bt["strategy_ret"].std() + 1e-12
sharpe = float((strat_mean / strat_std) * np.sqrt(ann_factor))

cum_return = float(bt["equity_strategy"].iloc[-1] - 1)
mdd = max_drawdown(bt["equity_strategy"])

print(f"Strategy cumulative return: {cum_return:.4f}")
print(f"Strategy Sharpe (approx): {sharpe:.4f}")
print(f"Strategy max drawdown: {mdd:.4f}")

plt.figure(figsize=(12, 5))
plt.plot(bt["timestamp"], bt["equity_market"], label="Buy & Hold")
plt.plot(bt["timestamp"], bt["equity_strategy"], label="Model strategy")
plt.title("Equity curve comparison")
plt.legend()
plt.grid(True)
plt.show()

## 11) Inference on the latest candle

In [ ]:
latest = X.iloc[[-1]]
p_next_close = float(xgb_reg.predict(latest)[0])
p_up_prob = float(rf_clf.predict_proba(latest)[0, 1])
p_up = int(p_up_prob > 0.5)

print("Current close:", round(float(model_df['close'].iloc[-1]), 4))
print("Pred next close:", round(p_next_close, 4))
print("Pred prob(up):", round(p_up_prob, 4))
print("Pred direction up=1/down=0:", p_up)

## 12) Save artifacts for reproducibility

In [ ]:
artifact = {
    "data_source": DATA_SOURCE,
    "symbol": SYMBOL,
    "yf_ticker": YF_TICKER,
    "timeframe": TIMEFRAME,
    "feature_cols": feature_cols,
    "direction_threshold": DIRECTION_THRESHOLD,
    "scaler": scaler,
    "rf_clf": rf_clf,
    "xgb_reg": xgb_reg,
    "seed": SEED,
}
joblib.dump(artifact, "crypto_next_candle_artifact.joblib")
print("Saved artifact: crypto_next_candle_artifact.joblib")

In [ ]:
# Freeze key package versions (optional)
import importlib.metadata as im
pkgs = ["pandas", "numpy", "scikit-learn", "xgboost", "ta", "ccxt", "yfinance"]
for p in pkgs:
    try:
        print(f"{p}=={im.version(p)}")
    except Exception:
        pass

## 13) Optional Deep Learning extension (LSTM)
If you want sequence modeling:
1. Build rolling windows of past `N` candles.
2. Scale per-feature on training split only.
3. Train LSTM/GRU (TensorFlow or PyTorch) on train, validate on later segment.
4. Keep chronological split and compare to ML baselines.

You can add TensorFlow in Colab with:
```python
!pip -q install tensorflow
```

## Suggested next upgrades
- Add transaction costs/slippage and position sizing.
- Calibrate decision threshold on validation, not test.
- Add feature selection (correlation filter, RFE, SHAP).
- Add regime filters (trend/volatility states).
- Run walk-forward retraining loops for production realism.
- Re-run the notebook top-to-bottom after `Factory reset runtime`.